In [1]:
# import libraries
import mne, os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib
# import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')

In [2]:
subj = 'sub-LTP327'
main_dir = '/Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/'
raw_dir = os.path.join(main_dir, 'raw', subj)
data_dir = os.path.join(main_dir, 'data', subj)

# load data

In [3]:
task = 'task-ltpFR2'
sess = 'ses-0'

# HyrdoCel Geodesic
# load eeg data
fname = os.path.join(raw_dir, '%s', 'eeg', '%s_%s_%s_eeg.edf')%(sess, subj, sess, task)
raw = mne.io.read_raw_edf(fname, preload=True, eog=['E8', 'E25', 'E126', 'E127'])
# raname channels
raw.rename_channels({'E129': 'Cz'})

# set up sensor layout
layout = mne.channels.make_standard_montage('GSN-HydroCel-129')
raw.set_montage(layout, on_missing='warn')

Extracting EDF parameters from /Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/raw/sub-LTP327/ses-0/eeg/sub-LTP327_ses-0_task-ltpFR2_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2935999  =      0.000 ...  5871.998 secs...


/var/folders/0c/37y3wbss32lb5vm9bhwy2xd40000gr/T/ipykernel_66767/729178790.py:13: RuntimeWarning: Not setting positions of 4 eog channels found in montage:
['E8', 'E25', 'E126', 'E127']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw.set_montage(layout, on_missing='warn')


<RawEDF | sub-LTP327_ses-0_task-ltpFR2_eeg.edf, 129 x 2936000 (5872.0 s), ~2.82 GiB, data loaded>

# re-reference data

In [4]:
# re-reference to average channel
data_reref, avg_ref = mne.set_eeg_reference(raw, ref_channels='average', copy=True, ch_type=['eeg'])

# re-reference EOG channels by subtracting "avg_ref"
# eog=['E8', 'E25', 'E126', 'E127']
data_reref._data[7,:] -= avg_ref
data_reref._data[24,:] -= avg_ref
data_reref._data[125,:] -= avg_ref
data_reref._data[126,:] -= avg_ref

Applying average reference.
Applying a custom ('EEG',) reference.


# identify bad channels

In [5]:
data_reref.plot(n_channels=100, duration=30, color='steelblue',
                # events=events, event_color='orange', 
                highpass=1, lowpass=40, scalings=dict(eeg=2e-5)
               )
data_reref.plot_psd(fmax=50)

Setting up band-pass filter from 1 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 1.00, 40.00 Hz: -6.02, -6.02 dB

Using matplotlib as 2D backend.
NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Effective window size : 4.096 (s)
Plotting power spectral density (dB=True).


<MNELineFigure size 1000x350 with 2 Axes>

In [7]:
# check channels maked as bad
data_reref.info['bads']

[np.str_('E43'), np.str_('E105'), np.str_('E15')]

# interpolate bad channel

In [8]:
# original raw data
print(raw.info['bads'])

# update bad channels info to the orginal raw data 
raw.info['bads'] = data_reref.info['bads']
print(raw.info['bads'])

# interpolate bad channels
data_interp = raw.copy()
data_interp.interpolate_bads(reset_bads=True, method=dict(eeg='spline'))

[]
[np.str_('E43'), np.str_('E105'), np.str_('E15')]
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 94.0 mm
Computing interpolation matrix from 122 sensor positions
Interpolating 3 sensors


<RawEDF | sub-LTP327_ses-0_task-ltpFR2_eeg.edf, 129 x 2936000 (5872.0 s), ~2.82 GiB, data loaded>

# re-reference data again

In [9]:
# re-reference to average channel
data_reref, avg_ref = mne.set_eeg_reference(data_interp, ref_channels='average', copy=True, ch_type=['eeg'])

# re-reference EOG channels by subtracting "avg_ref"
# eog=['E8', 'E25', 'E126', 'E127']
data_reref._data[7,:] -= avg_ref
data_reref._data[24,:] -= avg_ref
data_reref._data[125,:] -= avg_ref
data_reref._data[126,:] -= avg_ref

Applying average reference.
Applying a custom ('EEG',) reference.


# annotating movement artifacts as bad spans

In [10]:
data_reref.plot(n_channels=100, duration=30, color='steelblue',
                # events=events, event_color='orange', 
                highpass=1, lowpass=40, scalings=dict(eeg=2e-5)
               )

Setting up band-pass filter from 1 - 40 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 1.00, 40.00 Hz: -6.02, -6.02 dB



<MNEBrowseFigure size 2066x1296 with 4 Axes>

Channels marked as bad:
none


In [11]:
# save annotation log file
annot_fname = os.path.join(data_dir, '%s', '%s_%s_annot.csv') %(sess, subj, sess)
data_reref.annotations.save(annot_fname, overwrite=True)

print('save annotations')
print(data_reref.annotations)

# # read annotation log file
# annot_fname = os.path.join(data_dir, '%s', '%s_%s_annot.csv') %(sess, subj, sess)

# if os.path.exists(annot_fname):
#     annot_badspan = mne.read_annotations(annot_fname)
#     data_reref.set_annotations(annot_badspan, emit_warning=False)
#     print('read annotations')
#     print(data_reref.annotations)
# else:
#     print('no annotation file!')

save annotations
<Annotations | 4 segments: BAD_ (4)>


# filtering data before ICA

In [12]:
data_filt = data_reref.copy()
data_filt.filter(l_freq=1, h_freq=40, method='fir')

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1651 samples (3.302 s)



<RawEDF | sub-LTP327_ses-0_task-ltpFR2_eeg.edf, 129 x 2936000 (5872.0 s), ~2.82 GiB, data loaded>

# Indepent Component Analysis

In [13]:
ica = mne.preprocessing.ICA(n_components=None, method='fastica', random_state=1119)
ica.fit(data_filt, reject_by_annotation=True)

Fitting ICA to data using 125 channels (please be patient, this may take a while)
Omitting 267244 of 2936000 (9.10%) samples, retaining 2668756 (90.90%) samples.
Selecting by non-zero PCA components: 113 components


/var/folders/0c/37y3wbss32lb5vm9bhwy2xd40000gr/T/ipykernel_66767/871854609.py:2: RuntimeWarning: Using n_components=None (resulting in n_components_=113) may lead to an unstable mixing matrix estimation because the ratio between the largest (57) and smallest (1.8e-05) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 85
  ica.fit(data_filt, reject_by_annotation=True)


Fitting ICA took 245.7s.


Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,112 iterations on raw data (2668756 samples)
ICA components,113
Available PCA components,125
Channel types,eeg
ICA components marked for exclusion,—


In [14]:
# save ica file
ica_fname = os.path.join(data_dir, '%s', '%s_%s-ica.fif') %(sess, subj, sess)
ica.save(ica_fname, overwrite=True)

# # read ica file
# ica_fname = os.path.join(data_dir, '%s', '%s_%s-ica.fif') %(sess, subj, sess)
# ica = mne.preprocessing.read_ica(ica_fname)

Writing ICA solution to /Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/data/sub-LTP327/ses-0/sub-LTP327_ses-0-ica.fif...


Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,112 iterations on raw data (2668756 samples)
ICA components,113
Available PCA components,125
Channel types,eeg
ICA components marked for exclusion,—


In [15]:
# plot ICA source waveform
ica.plot_sources(data_filt)

# plot the first 40 ICA components
ica.plot_components(picks=np.arange(0,50), nrows=10, ncols=5)

Creating RawArray with float64 data, n_channels=117, n_times=2936000
    Range : 0 ... 2935999 =      0.000 ...  5871.998 secs
Ready.


<MNEFigure size 975x1299 with 50 Axes>

In [16]:
# print marked ICA components
print(ica.exclude)

[6, 43]


In [17]:
# apply ICA to the re-reference data (without filtering)
data_ica = ica.apply(data_reref, exclude=ica.exclude)

Applying ICA to Raw instance
    Transforming to ICA space (113 components)
    Zeroing out 2 ICA components
    Projecting back using 125 PCA components


In [18]:
# save raw file after removing ica components
ica_fname = os.path.join(data_dir, '%s', '%s_%s_ICA-raw.fif') %(sess, subj, sess)
data_ica.save(ica_fname, overwrite=True)

Writing /Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/data/sub-LTP327/ses-0/sub-LTP327_ses-0_ICA-raw.fif
Closing /Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/data/sub-LTP327/ses-0/sub-LTP327_ses-0_ICA-raw.fif
[done]


[PosixPath('/Users/cl2030/Library/CloudStorage/OneDrive-RutgersUniversity/projects/PEERS/data/sub-LTP327/ses-0/sub-LTP327_ses-0_ICA-raw.fif')]